# Fourier Neural Operator 学习一维 Burgers 方程

这个 Notebook 用于项目展示：我们使用 PyTorch 实现一个简洁的 `FNO1d`，学习一维黏性 Burgers 方程中的函数到函数映射。

展示目标：

- 从数值解数据构造监督学习样本：`u(x, 0) -> u(x, T)`。
- 训练 Fourier Neural Operator，并与一个简单 1D CNN baseline 做误差比较。
- 可视化预测解与真实数值解，观察模型在 PDE 建模任务中的表现。
- 通过这个小项目理解微分方程、数值模拟和深度学习算子学习之间的联系。

## 1. PDE 背景

一维黏性 Burgers 方程为：

$$u_t + u u_x = \nu u_{xx}$$

其中：

- `u(x,t)` 是位置 `x` 和时间 `t` 上的物理场。
- `u u_x` 表示非线性对流项。
- `nu u_xx` 表示黏性扩散项。
- 本项目使用周期边界条件，空间区域为 `x in [0, 1]`。

神经算子学习的目标不是预测单个标签，而是学习一个从函数到函数的映射：

$$G: u(x,0) \mapsto u(x,T)$$

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.burgers import load_burgers_splits
from src.evaluation.evaluate import load_model_checkpoint
from src.utils.config import load_config

config = load_config(PROJECT_ROOT / "configs/default.yaml")
paths = config["paths"]

dataset_path = PROJECT_ROOT / paths["data_dir"] / paths["dataset_file"]
output_dir = PROJECT_ROOT / paths["output_dir"]
checkpoint_dir = PROJECT_ROOT / paths["checkpoint_dir"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {dataset_path}")

## 2. 从数值解生成训练数据

项目使用随机平滑周期初值作为 `u(x,0)`，再用一个小型 RK4 数值求解器推进 Burgers 方程，得到 `u(x,T)`。

从仓库根目录重新生成数据：

```bash
python scripts/generate_data.py --config configs/default.yaml
```

默认配置很小，适合 CPU 演示：64 个训练样本、16 个验证样本、16 个测试样本，空间网格为 32 点。

In [ ]:
splits = load_burgers_splits(dataset_path)

for split_name, (inputs, targets) in splits.items():
    print(f"{split_name:>5}: inputs={tuple(inputs.shape)}, targets={tuple(targets.shape)}")

data_config = config["data"]
print("\nPDE/data config:")
for key in ["n_grid", "viscosity", "time_horizon", "time_steps", "initial_condition_modes"]:
    print(f"  {key}: {data_config[key]}")

In [ ]:
test_inputs, test_targets = splits["test"]
sample_index = 0

u0 = test_inputs[sample_index].numpy()
uT = test_targets[sample_index].numpy()
x = np.linspace(0.0, 1.0, len(u0), endpoint=False)

plt.figure(figsize=(8, 4))
plt.plot(x, u0, "--", label="initial u(x, 0)", color="tab:gray")
plt.plot(x, uT, label="numerical truth u(x, T)", color="black")
plt.xlabel("x")
plt.ylabel("u")
plt.title("One supervised Burgers sample")
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

## 3. FNO1d 模型思想

Fourier Neural Operator 的核心是频域卷积：先对空间维度做 FFT，只保留低频 Fourier modes，再回到物理空间。对周期问题来说，这和 PDE 中常见的谱方法有自然联系。

本项目的 `FNO1d` 做了三件事：

1. 将每个网格点的 `u(x)` 和坐标 `x` lift 到更高维特征。
2. 多次交替使用 spectral convolution 和 pointwise convolution。
3. 将隐藏特征 project 回一维场 `u(x,T)`。

模型实现位置：`src/models/fno1d.py`。

In [ ]:
device = torch.device(config.get("device", "cpu"))
fno_model, fno_checkpoint = load_model_checkpoint(
    checkpoint_dir / paths["checkpoint_file"],
    device,
    model_name="fno",
)

n_parameters = sum(parameter.numel() for parameter in fno_model.parameters())
print(fno_model)
print(f"\nNumber of parameters: {n_parameters:,}")

## 4. 训练与评估命令

从仓库根目录可以按下面的顺序复现结果：

```bash
python scripts/generate_data.py --config configs/default.yaml
python scripts/train_fno.py --config configs/default.yaml
python scripts/train_baseline.py --config configs/default.yaml
python scripts/evaluate.py --config configs/default.yaml --model fno
python scripts/evaluate.py --config configs/default.yaml --model baseline
python scripts/plot_results.py --config configs/default.yaml
```

评价指标包括：

- MSE：逐点均方误差。
- relative L2：相对二范数误差，更适合比较函数预测整体形状。

In [ ]:
with (output_dir / paths["comparison_metrics_file"]).open("r", encoding="utf-8") as file:
    comparison = json.load(file)

print("Test-set metrics")
print("model        MSE          relative L2")
print("-------------------------------------")
for model_name, metrics in comparison.items():
    print(f"{model_name:<10} {metrics['mse']:<12.6f} {metrics['relative_l2']:.6f}")

print("\nNote: this tiny CPU-friendly config is for demonstration, not hyperparameter tuning.")

## 5. 预测结果可视化

下面的图比较了一个测试样本上的初始条件、真实数值解、FNO 预测和 CNN baseline 预测，并显示逐点绝对误差。

![Prediction comparison](../outputs/prediction_comparison.png)

## 6. 不同时间步下的滚动预测

当前训练任务是单步算子 `u(x,t) -> u(x,t+T)`。为了展示不同时间步，我们将训练好的 FNO 连续应用三次，得到 `T, 2T, 3T` 的滚动预测，并用数值求解器生成同一时间点的真实解。

这个实验是定性展示：误差通常会随着滚动步数增加而累积。

![FNO rollout comparison](../outputs/fno_rollout_comparison.png)

## 7. 训练曲线

训练曲线可以帮助判断模型是否在学习、验证误差是否下降，以及小数据设置下是否存在过拟合迹象。

![Loss curves](../outputs/loss_curves.png)

## 8. 小结

这个项目展示了一个完整的 PDE operator learning 最小工作流：

- 用数值方法生成 Burgers 方程数据。
- 用 PyTorch 实现 FNO1d，学习从初始函数到未来函数的映射。
- 用 MSE 和 relative L2 评估预测误差。
- 用预测曲线、误差曲线和滚动预测图观察模型行为。

当前默认配置偏向快速演示。后续可以继续改进：增加数据量、提高网格分辨率、训练更久、做超参搜索，或者将任务扩展为直接预测完整时间轨迹 `u(x,t)`。